# 🎮 Multiplayer Balance & Matchmaking Insights
Дашборд контроля качества мультиплеера: матчмейкинг, баланс карт, безопасность и баланс героев.
Фильтры по региону сервера и карте применяются к вкладкам. Запуск: `voila notebooks/dashboard.ipynb --port=8866`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

pio.renderers.default = 'notebook'
pio.templates.default = 'plotly_white'
import plotly.graph_objects as _go
W = 1150

matches = pd.read_csv('../data/matches.csv', keep_default_na=False)  # регион 'NA' — не пропуск
stats = pd.read_csv('../data/player_stats.csv')
reports = pd.read_csv('../data/reports.csv')

pm = stats.merge(matches, on='match_id')
pm['win'] = (pm['team_side'] == pm['winning_team']).astype(int)
pm['kd_ratio'] = pm['kills'] / np.where(pm['deaths'] == 0, 1, pm['deaths'])

team = pm.groupby(['match_id', 'team_side'])['mmr_before'].agg(['mean', 'std']).unstack()
team.columns = [f'{a}_{b}' for a, b in team.columns]
mm = matches.set_index('match_id').join(team)
mm['mmr_delta'] = (mm['mean_Attack'] - mm['mean_Defense']).abs()
mm['team_std'] = mm[['std_Attack', 'std_Defense']].max(axis=1)
mm['stronger_won'] = np.where(mm['mean_Attack'] >= mm['mean_Defense'],
                              mm['winning_team'] == 'Attack', mm['winning_team'] == 'Defense')

rec = reports.groupby(['reported_id', 'reason']).size().unstack(fill_value=0)
rec['reports_total'] = rec.sum(axis=1)
prof = pm.groupby('player_id').agg(matches=('match_id', 'nunique'), avg_kd=('kd_ratio', 'mean'),
                                   avg_mmr=('mmr_before', 'mean'), minutes=('duration_sec', lambda x: x.sum() / 60))
prof = prof.join(rec).fillna(0)
prof['report_density'] = prof['reports_total'] / prof['minutes'] * 100
vet = prof[prof['matches'] >= 15].copy()
vet['toxicity_density'] = vet['toxicity'] / vet['minutes'] * 100
vet['flag'] = np.select(
    [vet['avg_kd'] > 4.5, vet['avg_kd'].between(2.5, 4.5) & (vet['avg_mmr'] < 1600), vet['toxicity_density'] > 0.5],
    ['Cheating → Sent_to_Overwatch', 'Smurf', 'Toxicity'], '')
cand = vet[vet['flag'] != ''].sort_values(['avg_kd', 'reports_total'], ascending=False)

region_w = widgets.Dropdown(options=['Все'] + sorted(matches['server_region'].unique()), value='Все', description='Регион:')
map_w = widgets.Dropdown(options=['Все'] + sorted(matches['map_id'].unique()), value='Все', description='Карта:')

def mfilter(df):
    if region_w.value != 'Все':
        df = df[df['server_region'] == region_w.value]
    if map_w.value != 'Все':
        df = df[df['map_id'] == map_w.value]
    return df

def kpi(title, value, color='#3B8EA5'):
    return (f'<div style="flex:1;border:1px solid #ddd;border-radius:10px;padding:12px 16px;background:#fff">'
            f'<div style="color:#666;font-size:13px">{title}</div>'
            f'<div style="font-size:24px;font-weight:bold;color:{color}">{value}</div></div>')

In [ ]:
# ---------- Вкладка 1: Матчмейкинг ----------
out_mm = widgets.Output()
def render_mm(*_):
    d = mfilter(mm.reset_index())
    with out_mm:
        clear_output(wait=True)
        unfair = (d['mmr_delta'] > 250).mean() * 100
        display(HTML('<div style="display:flex;gap:12px;margin:6px 0">'
                     + kpi('Матчей', f"{len(d):,}")
                     + kpi('Медиана ΔMMR', f"{d['mmr_delta'].median():.0f}")
                     + kpi('Unfair_Match (ΔMMR > 250)', f"{unfair:.1f}%", '#C62828' if unfair > 10 else '#2E7D32')
                     + kpi('Win Rate сильной команды', f"{d['stronger_won'].mean() * 100:.1f}%") + '</div>'))
        pct = d['mmr_delta'].quantile([0.5, 0.75, 0.9, 0.95, 0.99])
        fig = go.Figure(go.Histogram(x=d['mmr_delta'], nbinsx=60, marker_color='#3B8EA5', name='Матчи'))
        for q, v in pct.items():
            fig.add_vline(x=v, line_dash='dot', line_color='gray', annotation_text=f'P{int(q * 100)}={v:.0f}',
                          annotation_textangle=-90)
        fig.add_vline(x=250, line_color='red', annotation_text='Unfair 250')
        fig.update_layout(title='Распределение матчей по ΔMMR (с перцентилями)', xaxis_title='ΔMMR',
                          yaxis_title='Матчей', height=380, margin=dict(t=50, b=30), width=W)
        fig.show()
        d = d.assign(qbin=pd.qcut(d['queue_time_sec'], 10, duplicates='drop'))
        q = d.groupby('qbin', observed=True).agg(queue=('queue_time_sec', 'median'), delta=('mmr_delta', 'mean'),
                                                 std=('team_std', 'mean'))
        fig = go.Figure([go.Scatter(x=q['queue'], y=q['std'], mode='lines+markers', name='StDev MMR в команде'),
                         go.Scatter(x=q['queue'], y=q['delta'], mode='lines+markers', name='ΔMMR')])
        fig.update_layout(title='Queue Time vs Match Quality', xaxis_title='Время поиска, сек (медиана дециля)',
                          yaxis_title='MMR, пункты', height=360, margin=dict(t=50, b=30), width=W)
        fig.show()

# ---------- Вкладка 2: Баланс карт ----------
out_map = widgets.Output()
def render_map(*_):
    d = matches if region_w.value == 'Все' else matches[matches['server_region'] == region_w.value]
    mb = d.groupby('map_id')['winning_team'].value_counts(normalize=True).unstack().fillna(0) * 100
    mb['n'] = d['map_id'].value_counts()
    mb = mb.sort_values('Attack')
    risk = mb[['Attack', 'Defense']].max(axis=1) > 55
    with out_map:
        clear_output(wait=True)
        fig = go.Figure([
            go.Bar(y=mb.index, x=mb['Attack'], orientation='h', name='Attack',
                   marker_color=['#C62828' if r else '#3B8EA5' for r in risk],
                   text=[f'Attack {v:.1f}%' for v in mb['Attack']], textposition='inside'),
            go.Bar(y=mb.index, x=mb['Defense'], orientation='h', name='Defense',
                   marker_color=['#EF9A9A' if r else '#F6B26B' for r in risk],
                   text=[f'Defense {v:.1f}%' for v in mb['Defense']], textposition='inside')])
        fig.add_vrect(x0=48, x1=52, fillcolor='green', opacity=0.12, line_width=0)
        fig.add_vline(x=50, line_dash='dash')
        fig.update_layout(barmode='stack', title=f'Баланс сторон на картах (красный — сторона > 55%), регион: {region_w.value}',
                          xaxis=dict(range=[0, 100], ticksuffix='%'), height=380, margin=dict(t=50, b=30), width=W)
        fig.show()
        tbl = mb.round(1).assign(status=np.where(risk, '🔴 Map_Disedged', '🟢 OK'))
        display(tbl)

# ---------- Вкладка 3: Безопасность ----------
out_sec = widgets.Output()
reason_w = widgets.Dropdown(options=['Все'] + sorted(cand['flag'].unique()), value='Все', description='Причина:')
player_w = widgets.Dropdown(options=list(cand.index), description='Игрок:')
out_log = widgets.Output()
def render_sec(*_):
    c = cand if reason_w.value == 'Все' else cand[cand['flag'] == reason_w.value]
    with out_sec:
        clear_output(wait=True)
        display(HTML(f'<b>Кандидатов на проверку: {len(c)}</b> (стаж ≥ 15 матчей; сортировка по K/D и жалобам)'))
        display(c[['matches', 'avg_kd', 'avg_mmr', 'reports_total', 'cheating', 'toxicity', 'report_density', 'flag']]
                .head(25).style.format(precision=2).background_gradient(subset=['avg_kd', 'reports_total'], cmap='Reds'))
    player_w.options = list(c.index) or ['—']
def render_log(*_):
    with out_log:
        clear_output(wait=True)
        pid = player_w.value
        if pid not in prof.index:
            return
        log = pm[pm['player_id'] == pid].sort_values('match_date')
        log = log.assign(reports=log['match_id'].map(reports[reports['reported_id'] == pid].groupby('match_id').size()).fillna(0))
        display(HTML(f'<b>Лог матчей игрока {pid}</b>: {len(log)} матчей, средний K/D {log["kd_ratio"].mean():.2f}'))
        display(log[['match_date', 'match_id', 'map_id', 'team_side', 'hero_id', 'kills', 'deaths', 'kd_ratio',
                     'win', 'mmr_before', 'reports']].round(2).head(20).reset_index(drop=True))

# ---------- Вкладка 4: Баланс героев ----------
out_hero = widgets.Output()
COLORS = {'Критический Нерф (Имба)': '#ef553b', 'Требуется Бафф (Популярен, но слаб)': '#ab63fa',
          'Норма: Ситуативный (Counter-pick)': '#19d3f3', 'Норма: Непопулярный / Слабый': '#fecb52',
          'Идеальный баланс': '#00cc96'}
def render_hero(*_):
    d = mfilter(pm)
    d = d.assign(mirror=d.groupby(['match_id', 'hero_id'])['team_side'].transform('nunique') > 1)
    per = d.drop_duplicates(['match_id', 'team_side', 'hero_id'])
    h = per.groupby('hero_id').agg(Matches_Played=('match_id', 'nunique'))
    h['Pick_Rate'] = h['Matches_Played'] / d['match_id'].nunique() * 100
    h['Clean_Win_Rate'] = per[~per['mirror']].groupby('hero_id')['win'].mean() * 100
    avg = h['Pick_Rate'].mean()
    h['Status'] = np.select(
        [(h['Clean_Win_Rate'] > 52) & (h['Pick_Rate'] > avg), (h['Clean_Win_Rate'] < 48) & (h['Pick_Rate'] > avg),
         (h['Clean_Win_Rate'] > 52), (h['Clean_Win_Rate'] < 48)],
        list(COLORS)[:4], 'Идеальный баланс')
    with out_hero:
        clear_output(wait=True)
        fig = px.scatter(h.reset_index(), x='Pick_Rate', y='Clean_Win_Rate', color='Status', text='hero_id',
                         size='Matches_Played', size_max=30, color_discrete_map=COLORS, hover_name='hero_id',
                         hover_data={'Pick_Rate': ':.1f', 'Clean_Win_Rate': ':.2f', 'Matches_Played': True,
                                     'hero_id': False, 'Status': False},
                         labels={'Pick_Rate': 'Pick Rate, %', 'Clean_Win_Rate': 'Чистый Win Rate, %'},
                         title=f'Матрица баланса героев (регион: {region_w.value}, карта: {map_w.value})')
        fig.update_traces(textposition='top center')
        for y, c, t in [(50, 'black', '50%'), (52, 'red', '52%'), (48, 'blue', '48%')]:
            fig.add_hline(y=y, line_color=c, opacity=0.4, line_dash='dash' if y == 50 else 'solid', annotation_text=t)
        fig.add_vline(x=avg, line_dash='dot', line_color='gray', annotation_text=f'Ср. Pick {avg:.1f}%')
        fig.update_layout(yaxis=dict(range=[40, 60], ticksuffix='%'), xaxis=dict(ticksuffix='%'),
                          template='plotly_white', height=560, legend_title_text='Решение гейм-дизайнера', width=W)
        fig.show()

def render_all(*_):
    render_mm(); render_map(); render_sec(); render_log(); render_hero()

region_w.observe(render_all, names='value')
map_w.observe(lambda *_: (render_mm(), render_hero()), names='value')
reason_w.observe(lambda *_: (render_sec(), render_log()), names='value')
player_w.observe(render_log, names='value')

tabs = widgets.Tab(children=[
    out_mm, out_map,
    widgets.VBox([reason_w, out_sec, widgets.HTML('<h4>Детализация: лог матчей игрока</h4>'), player_w, out_log]),
    out_hero])
for i, t in enumerate(['Матчмейкинг', 'Баланс карт', 'Безопасность', 'Баланс героев']):
    tabs.set_title(i, t)
display(widgets.HBox([region_w, map_w]), tabs)
render_all()